# ITW Row-Mask Generator — fastMRI (nested D3PM priors)

Trains a scout-conditioned **Cartesian row mask** generator under a k-space row budget.

**Default objective (`mask_objective="nested_d3pm"`):** minimize nested frozen D3PM
conditional-entropy proxies — coarse PE-row profile + fine 96×96 magnitude — plus sparsity.

**Baseline:** set `mask_objective="infonce"` for the InfoNCE path (separate save dir).

Pretrain priors first (once):
```bash
uv run python d3pm_runner_fastmri.py --mode both --n-epochs 50
```

Do **not** load checkpoints from `models_mask_gen_fastmri/` (pre-fix / invalid survival tables).
Nested runs write to `models_mask_gen_fastmri_nested/`.

Acquisition: select rows in Fourier (k-space), reconstruct $Y = |\mathrm{ifft2}(X \odot k)|$.


In [ ]:
import torch

from itw import (
    FastMRIConfig,
    build_dataloader,
    build_mask_model,
    evaluate_fastmri_loader,
    evaluate_fastmri_nested_ablation,
    evaluate_fastmri_nested_loader,
    load_d3pm_coarse,
    load_d3pm_fine,
    plot_fastmri_grid,
    random_row_mask_batch,
    save_eval_report,
    sparsity_to_timestep,
    train_fastmri,
)
from itw.masks import apply_kspace_row_mask, gumbel_row_mask
from itw.schedule import build_fine_survival_table, build_row_survival_table


In [ ]:
cfg = FastMRIConfig(
    device="cuda" if torch.cuda.is_available() else "cpu",
    mask_objective="nested_d3pm",  # or "infonce"
    n_epochs=30,
    batch_size=8,
    sparsity_min=0.1,
    sparsity_max=0.4,
    sparsity_loss_weight=50.0,
    entropy_alpha=1.0,
    entropy_beta=1.0,
    save_every=1,
)
model = build_mask_model(cfg)
dataloader = build_dataloader(cfg)
print(
    f"device={cfg.device}, objective={cfg.mask_objective}, save_dir={cfg.save_dir}, "
    f"files={len(dataloader.dataset)}, batches/epoch={len(dataloader)}"
)


In [ ]:
# nested_d3pm -> (model, survival_tables); infonce -> (model, proj_head)
result = train_fastmri(cfg, model=model, dataloader=dataloader)
if cfg.mask_objective == "nested_d3pm":
    model, tables = result
    fine_survival = tables["fine_survival"]
    coarse_survival = tables["coarse_survival"]
    proj_head = None
    print(
        "survival fine", float(fine_survival.min()), float(fine_survival.max()),
        "t(0.1/0.25/0.4)",
        sparsity_to_timestep(torch.tensor([0.1, 0.25, 0.4]), fine_survival.cpu(), cfg.n_t).tolist(),
    )
    print(
        "survival coarse", float(coarse_survival.min()), float(coarse_survival.max()),
        "t(0.1/0.25/0.4)",
        sparsity_to_timestep(torch.tensor([0.1, 0.25, 0.4]), coarse_survival.cpu(), cfg.n_t).tolist(),
    )
else:
    model, proj_head = result
    fine_survival = coarse_survival = None


In [ ]:
if cfg.mask_objective == "nested_d3pm":
    d3pm_fine = load_d3pm_fine(cfg)
    d3pm_coarse = load_d3pm_coarse(cfg)
    if fine_survival is None:
        fine_survival = build_fine_survival_table(
            d3pm_fine, dataloader, device=cfg.device, fine_size=cfg.fine_size,
            n_bins=cfg.num_classes, max_batches=cfg.schedule_calibration_batches,
        )
    if coarse_survival is None:
        coarse_survival = build_row_survival_table(
            d3pm_coarse, dataloader, device=cfg.device, n_bins=cfg.num_classes,
            max_batches=cfg.schedule_calibration_batches,
        )
    metrics = evaluate_fastmri_nested_loader(
        model, d3pm_fine, d3pm_coarse, cfg, dataloader,
        fine_survival, coarse_survival, max_batches=10, fixed_sparsity=0.25,
    )
    print(metrics)
    print(
        "density", metrics["mean_sparsity_learned"],
        "t_fine", metrics.get("t_fine_mean"),
        "t_coarse", metrics.get("t_coarse_mean"),
    )
    save_eval_report(metrics, f"{cfg.save_dir}/eval_nested.json")

    ablation = evaluate_fastmri_nested_ablation(
        model, d3pm_fine, d3pm_coarse, cfg, dataloader,
        fine_survival, coarse_survival,
        sparsities=(0.1, 0.25, 0.4), max_batches=5,
    )
    save_eval_report(ablation, f"{cfg.save_dir}/eval_ablation.json")
    print(ablation)
else:
    metrics = evaluate_fastmri_loader(
        model, proj_head, cfg, dataloader, max_batches=10, fixed_sparsity=0.25
    )
    print(metrics)
    save_eval_report(metrics, f"{cfg.save_dir}/eval.json")


In [ ]:
model.eval()
z, c, kspace = next(iter(dataloader))
sparsity = torch.full((z.shape[0],), 0.25)

with torch.no_grad():
    z_d = z.to(cfg.device)
    k_d = kspace.to(cfg.device)
    s_d = sparsity.to(cfg.device)
    row_logits = model(z_d, s_d)
    learned_rows = gumbel_row_mask(row_logits, temperature=0.5, hard=True)
    random_rows = random_row_mask_batch(z.shape[0], cfg.image_size, s_d, cfg.device)
    y_learned = apply_kspace_row_mask(k_d, learned_rows)
    y_random = apply_kspace_row_mask(k_d, random_rows)

print("learned density", float(learned_rows.mean()), "y mean", float(y_learned.mean()))
plot_fastmri_grid(z, c, learned_rows.cpu(), y_learned.cpu(), title="learned row mask")
plot_fastmri_grid(z, c, random_rows.cpu(), y_random.cpu(), title="random row mask")


In [ ]:
# Eval / visualize the frozen D3PM priors (not the row-mask policy).
# Learned-vs-random nested mask eval needs `train_fastmri` first;
# do not load `models_mask_gen_fastmri/` (pre-fix, invalid survival tables).
#
# Saves figures + JSON under models_d3pm_fastmri_fine/eval/
%run -i eval_fastmri_priors.py